# Búsqueda de arquitectura de las redes compactas (sección 14 de la guía)

FCN, U-Net y SegNet compactas son diseño propio, así que recorren el espacio obligatorio de la guía:
3 bloques, anchos {16, 32, 64, 128, 256} en todas las combinaciones y dropout 0,2 en todas las
posiciones, **1000 arquitecturas por familia**. Protocolo fijado antes de correr en
`docs/PROTOCOLO_BUSQUEDA_ARQUITECTURA_E2.md` y `config/busqueda_arquitectura_e2.json`.

| paso | qué hace | orden de magnitud |
|---|---|---|
| piloto | 1 época de 15 configuraciones a 256 y a 128 px para medir tiempo | 15-30 min |
| fidelidad | elige la fidelidad de la fase 1 con la regla del protocolo y el presupuesto de horas | segundos |
| ajuste previo | tasa de Adam con la referencia (64,64,64)/(0,1,0), 3 tasas × 3 familias | minutos |
| fase 1 | 1000 × 3 familias, semilla 97, 6 épocas, a la fidelidad elegida; + 40 × 3 con la semilla 98 | la mayor parte |
| finalistas | ruido por semilla, k y finalistas (solo validación) | segundos |
| fase 2 | k finalistas + referencia + heredada, semillas 11, 23, 37, 53, 71, hasta 30 épocas | horas |
| resumen | `RESUMEN.json`, `RESUMEN.md`, `fase1.csv` | segundos |

**Todo es exploratorio y solo de validación**: el test del benchmark ya fue observado y aquí no se carga.
El cuaderno fue generado por `scripts/crear_colab_busqueda_arquitectura_e2.py` y probado en CPU con pocas
imágenes; **no se ha ejecutado en Colab** por quien lo preparó.

Cada corrida se reanuda sola: la fase 1 por corrida terminada (`result.json`), la fase 2 por época
(`last.pt`). Si la sesión se corta, vuelve a ejecutar las celdas 1 a 5 y la del paso en curso.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import subprocess, sys, os, json, hashlib, zipfile, base64, datetime, time
subprocess.run(['nvidia-smi'], check=True)
subprocess.run(['mountpoint', '-q', '/content/drive'], check=True)
OUTPUT = Path('/content/drive/MyDrive/tesis_imo_e2_busqueda_arquitectura_v1')
OUTPUT.mkdir(parents=True, exist_ok=True)
(OUTPUT/'logs').mkdir(exist_ok=True)
print('Salida:', OUTPUT, '| CPU:', os.cpu_count())

In [ ]:
# Mismo paquete que el benchmark v2: datos train/val, código y fuentes de terceros.
EXPECTED_ZIP_SHA256 = '954f4fe5663d9fedd676dcee4f1f7fb9086a2a7f5c1907ce902b5df4c109cb49'
ZIP = next((p for p in [Path('/content/paquete_colab_e2_v2.zip'), Path('/content/drive/MyDrive/paquete_colab_e2_v2.zip')] if p.exists()), None)
if ZIP is None: raise FileNotFoundError('Sube paquete_colab_e2_v2.zip al panel de archivos o a la raíz de Drive.')
def sha_file(p):
    h = hashlib.sha256()
    with Path(p).open('rb') as f:
        for b in iter(lambda: f.read(1024*1024), b''): h.update(b)
    return h.hexdigest()
assert sha_file(ZIP) == EXPECTED_ZIP_SHA256, 'ZIP distinto del v2 del benchmark'
BASE = Path('/content/imo_e2_v2'); BASE.mkdir(exist_ok=True)
with zipfile.ZipFile(ZIP) as z:
    for n in z.namelist():
        if not (BASE/n).resolve().is_relative_to(BASE): raise ValueError('Ruta no válida en ZIP')
    z.extractall(BASE)
manifest = json.loads((BASE/'PACKAGE_MANIFEST.json').read_text())
for name, expected in manifest['files'].items():
    assert sha_file(BASE/name) == expected, name
PROJECT = BASE/'tesis_imo_para_codex'
DEPENDENCIAS = {"scripts/experimento_e2_colab.py": "e6b0cb2a06b2ec50592ebee30f9c0fc627b984d13845f5fbd4fa6b85bbd00f81", "scripts/modelos_e2.py": "5b74e36860a86921fddd53dc2072404267929c4c9c14d92441f2f7c4bdd9d4db", "scripts/dataset_ready.py": "31b9707837c82baef6f7adea7716f007c0d6b16056d8ed13912c14dc839e2d5f", "scripts/modelos_base.py": "a342f30c03a12975e14c923d10362f4369ba9a9eb8dd8ad3e054e0e47e6603b2"}
for rel, esperado in DEPENDENCIAS.items():
    assert sha_file(PROJECT/rel) == esperado, f'{rel} no es el del protocolo cerrado'
print('Paquete íntegro:', len(manifest['files']), 'archivos; dependencias verificadas')

In [ ]:
# Archivos nuevos de la búsqueda, incrustados y verificados (generados desde el repositorio).
INCRUSTADOS = {
"scripts/busqueda_arquitectura_e2.py": {
"sha256": "8e1928be1532f82e3e21f4e8757e93dbc551a7d4f7042b32bc295948e48ad68a",
"b64": "..."
},
"scripts/resumir_busqueda_arquitectura_e2.py": {
"sha256": "4ac3f6d770638bc876ca07bb5130ba3c45cf7b76c3ff8c238458824f00998a24",
"b64": "..."
},
"scripts/complemento_entreno_e2.py": {
"sha256": "dfe815ac6137705098f0d966b88d081b786449a66499117dccf8e7bfd1aa8bf8",
"b64": "..."
},
"config/busqueda_arquitectura_e2.json": {
"sha256": "c8bbbc81dfd6529eac7e46216bba0d5de2da7caf198179fbc73ce3819634271e",
"b64": "ew0KICAic2NoZW1hIjogMSwNCiAgImV4cGVyaW1lbnRvIjogImUyX2J1c3F1ZWRhX2FycXVpdGVjdHVyYV9jb21wYWN0YXNfdjEiLA0KICAiZmVjaGFfcHJvdG9jb2xvIjogIjIwMjYtMTEtMjIiLA0KICAiZXN0YWRvIjogIlByb3RvY29sbyBmaWphZG8gYW50ZXMgZGUgY29ycmVyLiBOaW5nw7puIG7Dum1lcm8gZGUgZXN0ZSBhcmNoaXZvIGVzIHVuIHJlc3VsdGFkbyBkZSBsYSBiw7pzcXVlZGEuIiwNCiAgImRvY3VtZW50byI6ICJkb2NzL1BST1RPQ09MT19CVVNRVUVEQV9BUlFVSVRFQ1RVUkFfRTIubWQiLA0KICAiY29udHJvbF9ndWlhIjogIlNlY2Npw7NuIDE0ICgxNC4xIGEgMTQuNykgeSBzZWNjacOzbiA0LjE6IHRvZG8gbW9kZWxvIHNpbiBhcnF1aXRlY3R1cmEgZGVmaW5pZGEgZW4gc3UgYXJ0w61jdWxvIHJlY29ycmUgZWwgZXNwYWNpbyBvYmxpZ2F0b3JpbyBkZSAxMDAwIGFycXVpdGVjdHVyYXMgZW4gZG9zIGZhc2VzLiBDaGVja2xpc3QgMzggeSBCMTQuIiwNCiAgImV4cG9zaWNpb25fdGVzdCI6ICJFbCB0ZXN0IGRlbCBiZW5jaG1hcmsgeWEgZnVlIG9ic2VydmFkby4gRXN0YSBiw7pzcXVlZGEgbm8gY2FyZ2EgZWwgdGVzdDogZWwgYWp1c3RlIHByZXZpbywgZWwgY3JpYmFkbywgbG9zIGZpbmFsaXN0YXMgeSBsYSBzZWxlY2Npw7NuIHVzYW4gc29sbyB2YWxpZGFjacOzbiwgeSB0b2RvIHNlIGRlY2xhcmEgZXhwbG9yYXRvcmlvLiBFbCBwYXNvIEE2IGRlIGxhIGd1w61hICh1bmEgc29sYSBwYXNhZGEgcG9yIHRlc3QpIG5vIHNlIGhhY2UgZW4gZXN0YSBiw7pzcXVlZGEuIiwNCiAgInByb3RvY29sb19iYXNlIjogImNvbmZpZy9leHBlcmltZW50b19lMi5qc29uIiwNCiAgImZhbWlsaWFzIjogew0KICAgICJmY24iOiAiQ29kaWZpY2Fkb3IgZGUgMyBibG9xdWVzIGNvbnZvbHVjaW9uYWxlcyAoY29udiAzeDMgKyBSZUxVLCBkb3MgdmVjZXMpIGNvbiBtYXggcG9vbGluZyBlbnRyZSBibG9xdWVzOyBjYWJlemFsIDF4MSBzb2JyZSBlbCBibG9xdWUgMyBlIGludGVycG9sYWNpw7NuIGJpbGluZWFsIGEgbGEgcmVzb2x1Y2nDs24gZGUgZW50cmFkYS4gU2luIGRlY29kaWZpY2Fkb3IuIiwNCiAgICAidW5ldCI6ICJDb2RpZmljYWRvciBkZSAzIG5pdmVsZXM7IGRlY29kaWZpY2Fkb3IgZXNwZWpvIGRlIDIgYmxvcXVlcyBxdWUgY29uY2F0ZW5hIGVsIG1hcGEgZGVsIGNvZGlmaWNhZG9yIGRlbCBtaXNtbyBuaXZlbCAoZDI6IHczK3cyIC0+IHcyOyBkMTogdzIrdzEgLT4gdzEpOyBjYWJlemFsIDF4MS4iLA0KICAgICJzZWduZXQiOiAiQ29kaWZpY2Fkb3IgZGUgMyBuaXZlbGVzIGNvbiDDrW5kaWNlcyBkZSBtYXggcG9vbGluZzsgcmVkdWNjacOzbiAxeDEgdzMgLT4gdzI7IGRlY29kaWZpY2Fkb3IgZXNwZWpvIGNvbiBtYXggdW5wb29saW5nIChkMjogdzIgLT4gdzE7IGQxOiB3MSAtPiB3MSk7IGNhYmV6YWwgMXgxLiINCiAgfSwNCiAgImVzcGFjaW8iOiB7DQogICAgInByb2Z1bmRpZGFkIjogMywNCiAgICAiYW5jaG9zIjogWzE2LCAzMiwgNjQsIDEyOCwgMjU2XSwNCiAgICAicXVlX2VzX3dfaSI6ICJDYW5hbGVzIGRlIGxvcyBkb3MgY29udiAzeDMgZGVsIGJsb3F1ZSBkZWwgbml2ZWwgaSBkZWwgY29kaWZpY2Fkb3I7IGVsIGRlY29kaWZpY2Fkb3IgZXNwZWphIGxvcyBhbmNob3MgKHNlY2Npw7NuIDE0LjIsIGZpbGEgZGUgY29kaWZpY2Fkb3ItZGVjb2RpZmljYWRvcikuIiwNCiAgICAiZHJvcG91dF9wIjogMC4yLA0KICAgICJjYXBhX2Ryb3BvdXQiOiAidG9yY2gubm4uRHJvcG91dCAoZWxlbWVudG8gYSBlbGVtZW50byksIGxhIG1pc21hIGVuIGxhcyB0cmVzIGZhbWlsaWFzIiwNCiAgICAicG9zaWNpb25lc19kcm9wb3V0Ijogew0KICAgICAgImZjbiI6ICJiX2kgPSAxOiBkcm9wb3V0IHRyYXMgZWwgYmxvcXVlIGkgZGVsIGNvZGlmaWNhZG9yIChlbCBkZWwgYmxvcXVlIDMgcXVlZGEgYW50ZXMgZGVsIGNhYmV6YWwgMXgxKS4iLA0KICAgICAgInVuZXQiOiAiYl8xOiB0cmFzIGUxIHkgdHJhcyBzdSBlc3Blam8gZDE7IGJfMjogdHJhcyBlMiB5IHRyYXMgc3UgZXNwZWpvIGQyOyBiXzM6IHRyYXMgZTMsIHF1ZSBlcyBlbCBjdWVsbG8geSBubyB0aWVuZSBlc3Blam8uIiwNCiAgICAgICJzZWduZXQiOiAiYl8xOiB0cmFzIGUxIHkgdHJhcyBzdSBlc3Blam8gZDE7IGJfMjogdHJhcyBlMiB5IHRyYXMgc3UgZXNwZWpvIGQyOyBiXzM6IHRyYXMgZTMgKGFudGVzIGRlIGxhIHJlZHVjY2nDs24gMXgxKSwgY3VlbGxvIHNpbiBlc3Blam8uIg0KICAgIH0sDQogICAgImNhcmRpbmFsaWRhZCI6IDEwMDAsDQogICAgIm9yZGVuX2lkZW50aWZpY2Fkb3JlcyI6ICJQcm9kdWN0byBjYXJ0ZXNpYW5vIGl0ZXJ0b29scy5wcm9kdWN0KFcsIFcsIFcsIHswLDF9LCB7MCwxfSwgezAsMX0pOyBlbCBpZGVudGlmaWNhZG9yIGVzIGEwMDAwIGEgYTA5OTkgZW4gZXNlIG9yZGVuLiIsDQogICAgImxpc3RhZG8iOiAiY29uZmlnL2FycXVpdGVjdHVyYXNfY29tcGFjdGFzX2UyLmNzdiINCiAgfSwNCiAgInJlZmVyZW5jaWEiOiB7InciOiBbNjQsIDY0LCA2NF0sICJiIjogWzAsIDEsIDBdLCAicGFwZWwiOiAiQXJxdWl0ZWN0dXJhIGRlIHJlZmVyZW5jaWEgZGUgbGEgZ3XDrWE6IGFqdXN0ZSBwcmV2aW8geSBjb250cm9sIGVuIGxhIGZhc2UgMi4ifSwNCiAgImhlcmVkYWRhIjogeyJ3IjogWzE2LCAzMiwgNjRdLCAiYiI6IFswLCAwLCAwXSwgInBhcGVsIjogIkxhIGNvbXBhY3RhIGRlbCBiZW5jaG1hcmsuIEVzdMOhIGRlbnRybyBkZWwgZXNwYWNpbyB5IGVzIG51bcOpcmljYW1lbnRlIGlkw6ludGljYSBhIG1vZGVsb3NfYmFzZS5weSBjb24gbGEgbWlzbWEgc2VtaWxsYTsgZW50cmEgZW4gbGEgZmFzZSAyIGNvbW8gc2VndW5kbyBjb250cm9sIHBhcmEgbGEgY29tcGFyYWNpw7NuLiJ9LA0KICAicmVjZXRhIjogew0KICAgICJmdWVudGUiOiAiUmVjZXRhIGRlIGZjbiwgdW5ldCB5IHNlZ25ldCBlbiBjb25maWcvZXhwZXJpbWVudG9fZTIuanNvbiIsDQogICAgIm9wdGltaXplciI6ICJBZGFtIiwgImxyIjogMC4wMDEsICJ3ZWlnaHRfZGVjYXkiOiAwLjAsDQogICAgImJhdGNoX3NpemUiOiAyLCAic2l6ZSI6IDI1NiwgInBlcmRpZGEiOiAiMCw1IEJDRSArIDAsNSBEaWNlIHN1YXZlIHBvciBpbWFnZW4sIHNvbG8gZW4gbGEgem9uYSB2w6FsaWRhIiwNCiAgICAidW1icmFsIjogMC41LCAicHJlY2lzaW9uIjogImZsb2F0MzIgY29uIFRGMzIgZGUgY3VETk4gcG9yIGRlZmVjdG8geSBhbGdvcml0bW9zIGRldGVybWluaXN0YXMsIGNvbW8gZWwgYmVuY2htYXJrIiwNCiAgICAiY3JpdGVyaW9fdmFsaWRhY2lvbiI6ICJEaWNlIG1lZGlvIHBvciBncnVwbyBlbiB2YWxpZGFjacOzbiAobWVkaWEgZGUgaW3DoWdlbmVzIGRlbnRybyBkZWwgZ3J1cG8geSBsdWVnbyBlbnRyZSBncnVwb3MpIiwNCiAgICAiaW5pY2lhbGl6YWNpb24iOiAiTGEgZGUgUHlUb3JjaCBwb3IgZGVmZWN0bywgdHJhcyBzZXR1cF9zZWVkKHNlbWlsbGEpLCBpZ3VhbCBlbiB0b2RhcyBsYXMgYXJxdWl0ZWN0dXJhcyINCiAgfSwNCiAgImFqdXN0ZV9wcmV2aW8iOiB7DQogICAgImNvbnRyb2xfZ3VpYSI6ICIxNC4zOiBsb3MgaGlwZXJwYXLDoW1ldHJvcyBzZSBmaWphbiBhbnRlcyBkZSBsYSBiw7pzcXVlZGEgY29uIGxhIGFycXVpdGVjdHVyYSBkZSByZWZlcmVuY2lhICg2NCw2NCw2NCkgY29uIGRyb3BvdXQgKDAsMSwwKSwgdXNhbmRvIHNvbG8gdmFsaWRhY2nDs24uIiwNCiAgICAicXVlX3NlX2FqdXN0YSI6ICJTb2xvIGxhIHRhc2EgZGUgYXByZW5kaXphamUgZGUgQWRhbSwgcG9yIGZhbWlsaWE7IGVsIHJlc3RvIGRlIGxhIHJlY2V0YSBxdWVkYSBmaWpvLiIsDQogICAgImxyIjogWzAuMDAwMywgMC4wMDEsIDAuMDAzXSwNCiAgICAicHJlc3VwdWVzdG8iOiAiRWwgZGUgbGEgZmFzZSAxIChzZW1pbGxhLCDDqXBvY2FzIHkgcGFjaWVuY2lhKSIsDQogICAgInJlZ2xhIjogIlNlIHF1ZWRhIGxhIHRhc2EgY29uIG1heW9yIERpY2UgZGUgdmFsaWRhY2nDs24uIFNpIHN1IHZlbnRhamEgc29icmUgbGEgcmVjZXRhICgwLDAwMSkgZXMgbWVub3IgcXVlIDAsMDA1LCBzZSBxdWVkYSBsYSByZWNldGEuIExhIHRhc2EgZWxlZ2lkYSBxdWVkYSBmaWphIHBhcmEgbGFzIDEwMDAgYXJxdWl0ZWN0dXJhcyB5IHBhcmEgbGEgZmFzZSAyIGRlIGVzYSBmYW1pbGlhLiINCiAgfSwNCiAgImZhc2UxIjogew0KICAgICJzZW1pbGxhIjogOTcsDQogICAgImVwb2NhcyI6IDYsDQogICAgInBhY2llbmNpYSI6IDUsDQogICAgImZyYWNjaW9uX2RlX2Vwb2NhcyI6ICI2IGRlIDMwID0gMjAgJSwgZGVudHJvIGRlbCAyMC0zMCAlIHF1ZSBkYSBsYSBndcOtYSBjb21vIGVqZW1wbG8iLA0KICAgICJzZW1pbGxhX29yZGVuIjogMjAyNjExMjIsDQogICAgIm9yZGVuIjogIkxhcyAxMDAwIGFycXVpdGVjdHVyYXMgZW4gdW5hIHBlcm11dGFjacOzbiBmaWphIChzZW1pbGxhX29yZGVuKTsgcGFyYSBjYWRhIHVuYSwgbGFzIHRyZXMgZmFtaWxpYXMgc2VndWlkYXMuIEN1YWxxdWllciBwcmVmaWpvIGNvcnRhZG8gZXMgdW5hIG11ZXN0cmEgYWxlYXRvcmlhIGRlbCBlc3BhY2lvLCBpZ3VhbCBwYXJhIGxhcyB0cmVzIGZhbWlsaWFzLiIsDQogICAgIm1ldHJpY2EiOiAiTWVqb3IgRGljZSBkZSB2YWxpZGFjacOzbiBwb3IgZ3J1cG8gZW50cmUgbGFzIMOpcG9jYXMgY29ycmlkYXMiLA0KICAgICJyZWdpc3RybyI6ICJQb3Igw6lwb2NhOiBww6lyZGlkYSBkZSBlbnRyZW5hbWllbnRvLCBww6lyZGlkYSBkZSB2YWxpZGFjacOzbiB5IERpY2UgZGUgdmFsaWRhY2nDs24gcG9yIGdydXBvOyBwYXLDoW1ldHJvcywgTUFDcyB5IHNlZ3VuZG9zLiBObyBndWFyZGEgcGVzb3MuIiwNCiAgICAiZXhoYXVzdGl2YSI6ICJMYXMgMTAwMCBkZSBjYWRhIGZhbWlsaWEsIHNpbiBwb2RhIHByZXZpYSINCiAgfSwNCiAgImZpZGVsaWRhZF9mYXNlMSI6IHsNCiAgICAiY29udHJvbF9ndWlhIjogIjE0LjQ6IHNpIGVsIGPDs21wdXRvIG5vIGFsY2FuemEsIG5vIHNlIHJlY29ydGEgZW4gc2lsZW5jaW86IHNlIGRlY2xhcmEgbGEgZGVzdmlhY2nDs24sIHNlIGp1c3RpZmljYSB5IHNlIHJlcG9ydGEgc3UgZWZlY3RvLiBMYXMgMTAwMCBhcnF1aXRlY3R1cmFzLCBsYSBzZW1pbGxhIHkgbGFzIDYgw6lwb2NhcyBubyBjYW1iaWFuIGVuIG5pbmd1bmEgb3BjacOzbi4iLA0KICAgICJvcGNpb25lcyI6IFsNCiAgICAgIHsibm9tYnJlIjogIkEiLCAicHgiOiAyNTYsICJmcmFjY2lvbl90cmFpbiI6IDEuMCwgImRlc2NyaXBjaW9uIjogIkxhIGRlbCBwcm90b2NvbG86IGltYWdlbiBjb21wbGV0YSBhIDI1NiBweCwgdG9kbyB0cmFpbiJ9LA0KICAgICAgeyJub21icmUiOiAiQiIsICJweCI6IDEyOCwgImZyYWNjaW9uX3RyYWluIjogMS4wLCAiZGVzY3JpcGNpb24iOiAiTGEgcmVkIHZlIGxhIGltYWdlbiByZWR1Y2lkYSBhIDEyOCBweCBwb3IgcHJvbWVkaW8gZGUgw6FyZWE7IHN1IHNhbGlkYSB2dWVsdmUgYSAyNTYgcG9yIGludGVycG9sYWNpw7NuIGJpbGluZWFsIHkgbGEgcMOpcmRpZGEgeSBlbCBEaWNlIHNlIG1pZGVuIGEgMjU2IGNvbnRyYSBsYSBtw6FzY2FyYSBvcmlnaW5hbC4gVG9kbyB0cmFpbi4gQ8OzbXB1dG8gZGUgbGEgcmVkIC8gNCJ9LA0KICAgICAgeyJub21icmUiOiAiQyIsICJweCI6IDEyOCwgImZyYWNjaW9uX3RyYWluIjogMC41LCAiZGVzY3JpcGNpb24iOiAiQ29tbyBCLCBlbnRyZW5hbmRvIGNvbiBsYSBtaXRhZCBkZSB0cmFpbiBlbGVnaWRhIHBvciBncnVwb3MgZGUgc2ltaWxpdHVkIChzZW1pbGxhX3N1YmNvbmp1bnRvKSwgbGEgbWlzbWEgcGFyYSB0b2Rhcy4gVmFsaWRhY2nDs24gY29tcGxldGEifQ0KICAgIF0sDQogICAgInNlbWlsbGFfc3ViY29uanVudG8iOiAyMDI2MTEyMywNCiAgICAicmVnbGEiOiAiQ29uIGVsIHBpbG90byBlbiBMNCBzZSBlc3RpbWFuIGxhcyBob3JhcyBkZSBhanVzdGUgcHJldmlvICsgZmFzZSAxICsgcmVwZXRpY2nDs24gZGUgY2FkYSBvcGNpw7NuLiBTZSBlbGlnZSBsYSBwcmltZXJhIG9wY2nDs24sIGVuIGVsIG9yZGVuIEEsIEIsIEMsIGN1eWEgZXN0aW1hY2nDs24gbm8gcGFzZSBkZWwgcHJlc3VwdWVzdG8gZGUgaG9yYXMgcXVlIHNlIGZpamEgYW50ZXMgZGUgdmVyIGVsIHBpbG90by4gU2kgbmluZ3VuYSBjYWJlLCBzZSB1c2EgQyB5IHNlIHJlZ2lzdHJhIHF1ZSBleGNlZGUgZWwgcHJlc3VwdWVzdG8uIExhIGVsZWNjacOzbiBxdWVkYSBlbiBmaWRlbGlkYWQuanNvbiB5IG5vIHNlIGNhbWJpYSBkZXNwdcOpcyBkZSBlbXBlemFyIGVsIGFqdXN0ZSBwcmV2aW8uIExhIGZhc2UgMiBlcyBzaWVtcHJlIGEgZmlkZWxpZGFkIGNvbXBsZXRhLiIsDQogICAgImVmZWN0byI6ICJTZSByZXBvcnRhIGxhIGNvcnJlbGFjacOzbiBkZSByYW5nb3MgZW50cmUgbGEgZmFzZSAxIHkgbGEgZmFzZSAyIGVuIGxvcyBmaW5hbGlzdGFzOiBzaSBlcyBiYWphLCBlbCBjcmliYWRvIHJlZHVjaWRvIGZ1ZSBwb2NvIGluZm9ybWF0aXZvIHkgc2UgZGVjbGFyYS4iDQogIH0sDQogICJydWlkbyI6IHsNCiAgICAiY29udHJvbF9ndWlhIjogIjE0LjQ6IGVzdGltYXIgZWwgcnVpZG8gcG9yIHNlbWlsbGEgY29uIHVuIHBpbG90byBkZSBhcnF1aXRlY3R1cmFzIHJlcGV0aWRhcyB5IGVsZWdpciBrIGVuIGNvbnNlY3VlbmNpYTsgMTQuNjogcmVwZXRpciBsYSBmYXNlIDEgY29uIGFsIG1lbm9zIGRvcyBzZW1pbGxhcyBlbiB1bmEgbXVlc3RyYSByZXByZXNlbnRhdGl2YS4iLA0KICAgICJhcnF1aXRlY3R1cmFzIjogIkxhcyBwcmltZXJhcyA0MCBkZWwgb3JkZW4gZGUgbGEgZmFzZSAxIChtdWVzdHJhIGFsZWF0b3JpYSBkZWwgZXNwYWNpbyksIGVuIGxhcyB0cmVzIGZhbWlsaWFzIiwNCiAgICAibl9hcnF1aXRlY3R1cmFzIjogNDAsDQogICAgInNlbWlsbGFfZXh0cmEiOiA5OCwNCiAgICAiZXN0aW1hZG9yIjogInNpZ21hX3NlbWlsbGFeMiA9IG1lZGlhIGRlIChkXjIpLzIgY29uIGQgbGEgZGlmZXJlbmNpYSBlbnRyZSBsYXMgc2VtaWxsYXMgOTcgeSA5OCBkZSBsYSBtaXNtYSBhcnF1aXRlY3R1cmE7IHNpZ21hX2FycV4yID0gdmFyaWFuemEgZGUgbGFzIDEwMDAgZGUgbGEgZmFzZSAxIG1lbm9zIHNpZ21hX3NlbWlsbGFeMiAodHJ1bmNhZGEgZW4gMCk7IHIgPSBzaWdtYV9zZW1pbGxhIC8gc2lnbWFfYXJxLCBwb3IgZmFtaWxpYSIsDQogICAgInJlZ2xhX2siOiAiayA9IDEwIHNpIHIgPD0gMSBlbiBsYXMgdHJlcyBmYW1pbGlhczsgayA9IDIwIHNpIHIgPiAxIGVuIGFsZ3VuYS4gRWwgbWlzbW8gayBwYXJhIGxhcyB0cmVzICgxNC40OiBtaXNtbyB8QXwsIGsgeSBSKS4iDQogIH0sDQogICJmaW5hbGlzdGFzIjogew0KICAgICJvcmRlbiI6ICJEaWNlIGRlIGxhIGZhc2UgMSBkZXNjZW5kZW50ZTsgZW1wYXRlIGV4YWN0bywgbWVub3MgcGFyw6FtZXRyb3MiLA0KICAgICJjb250cm9sZXMiOiAiU2UgYcOxYWRlbiBsYSByZWZlcmVuY2lhICg2NCw2NCw2NCkvKDAsMSwwKSB5IGxhIGhlcmVkYWRhICgxNiwzMiw2NCkvKDAsMCwwKSBzaSBubyBlc3TDoW4gZW50cmUgbG9zIGsiLA0KICAgICJub3RhIjogIkxhIGZhc2UgMSBlcyB1biBjcmliYWRvOiBlbCBnYW5hZG9yIGVzIGVsIG1lam9yIGVudHJlIGxvcyBmaW5hbGlzdGFzLCBubyBuZWNlc2FyaWFtZW50ZSBlbCBtZWpvciBkZSBsYXMgMTAwMC4iDQogIH0sDQogICJmYXNlMiI6IHsNCiAgICAic2VtaWxsYXMiOiBbMTEsIDIzLCAzNywgNTMsIDcxXSwNCiAgICAibm90YV9zZW1pbGxhcyI6ICJMYXMgZGVsIGJlbmNobWFyaywgZGlzdGludGFzIGRlIGxhcyBkZSBsYSBmYXNlIDEgKDk3LCA5OCkuIENvbiBsYSBtaXNtYSBzZW1pbGxhIGVsIG9yZGVuIGRlIGxvdGVzIGVzIGVsIG1pc21vIHF1ZSBlbiBlbCBiZW5jaG1hcmssIGFzw60gcXVlIGxhIGhlcmVkYWRhIHNlIGNvbXBhcmEgcGFyZWFkYSBwb3Igc2VtaWxsYS4iLA0KICAgICJlcG9jYXMiOiAzMCwNCiAgICAicGFjaWVuY2lhIjogNSwNCiAgICAib3JkZW4iOiAiUG9yIHNlbWlsbGEgeSBsdWVnbyBwb3IgZmFtaWxpYSB5IGFycXVpdGVjdHVyYTogdW4gY29ydGUgZGVqYSBlbCBtaXNtbyBuw7ptZXJvIGRlIHNlbWlsbGFzIGVuIHRvZG9zIGxvcyBmaW5hbGlzdGFzIiwNCiAgICAicmVnaXN0cm8iOiAiUG9yIGNvcnJpZGE6IERpY2UgZGUgdmFsaWRhY2nDs24gcG9yIGdydXBvIGVuIGxhIG1lam9yIMOpcG9jYSwgcHJlZGljY2lvbmVzIHBvciBpbWFnZW4gZGUgdmFsaWRhY2nDs24sIERpY2UgZGUgZW50cmVuYW1pZW50byBwb3IgZ3J1cG8gY29uIGxvcyBwZXNvcyBkZSBsYSBtZWpvciDDqXBvY2EgKGJyZWNoYSB0cmFpbi12YWxpZGFjacOzbiksIHBhcsOhbWV0cm9zLCBNQUNzIHkgc2VndW5kb3MuIEd1YXJkYSBlbCBtZWpvciBjaGVja3BvaW50LiINCiAgfSwNCiAgInNlbGVjY2lvbiI6IHsNCiAgICAiY29udHJvbF9ndWlhIjogIjE0LjUiLA0KICAgICJkZWx0YSI6IDAuMDIsDQogICAgIm5vdGFfZGVsdGEiOiAizrQgPSAwLDAyIGRlIERpY2UsIGhlcmVkYWRvIGRlbCBwcmVyZWdpc3RybyIsDQogICAgInJlZ2xhIjogIihpKSBNYXlvciBtZWRpYSBkZSBEaWNlIGRlIHZhbGlkYWNpw7NuIGVuIGxhcyA1IHNlbWlsbGFzIGRlIGxhIGZhc2UgMi4gKGlpKSBFbXBhdGUgdMOpY25pY28gY29uIGVsIG1lam9yOiBkaWZlcmVuY2lhIGRlIG1lZGlhcyBtZW5vciBxdWUgzrQgbyBpbnRlcnZhbG9zIHQgYWwgOTUgJSBkZSBsYXMgbWVkaWFzIHNvbGFwYWRvcy4gRW50cmUgbG9zIGVtcGF0YWRvcyAoaW5jbHVpZG8gZWwgbWVqb3IpLCBsYSBkZSBtZW5vcyBwYXLDoW1ldHJvczsgc2kgZW1wYXRhbiBlbiBwYXLDoW1ldHJvcywgbGEgZGUgbWF5b3IgbWVkaWEuIExvcyBjb250cm9sZXMgKHJlZmVyZW5jaWEgeSBoZXJlZGFkYSkgY29tcGl0ZW4gY29tbyBjdWFscXVpZXIgZmluYWxpc3RhLiIsDQogICAgImluZGlzdGluZ3VpYmxlcyI6ICJUT1NUIHBhcmVhZG8gcG9yIHNlbWlsbGEgY29uIG1hcmdlbiDCsc60IGZyZW50ZSBhbCBtZWpvciAoZG9zIHQgdW5pbGF0ZXJhbGVzIGFsIDUgJSwgNCBnbCk7IHNlIHJlcG9ydGEgZWwgdGFtYcOxbyBkZWwgY29uanVudG8gZXF1aXZhbGVudGUiLA0KICAgICJwcm9oaWJpZG8iOiAiTWlyYXIgZWwgdGVzdCBwYXJhIGVsZWdpciBvIHJlYWJyaXIgbGEgYsO6c3F1ZWRhIGRlc3B1w6lzIGRlIHZlcmxvIg0KICB9LA0KICAiY29tcGFyYWNpb25faGVyZWRhZGEiOiB7DQogICAgInByZWd1bnRhIjogIsK/TGEgYXJxdWl0ZWN0dXJhIGdhbmFkb3JhIGRlIGNhZGEgZmFtaWxpYSBzdXBlcmEgYSBsYSBjb21wYWN0YSBoZXJlZGFkYSBkZWwgYmVuY2htYXJrLCBlbiB2YWxpZGFjacOzbj8iLA0KICAgICJjb250cmFzdGUiOiAiR2FuYWRvcmEgZnJlbnRlIGEgaGVyZWRhZGEsIGFtYmFzIGRlIGxhIGZhc2UgMiAobWlzbWEgcmVjZXRhLCBtaXNtYXMgNSBzZW1pbGxhcyk6IGRpZmVyZW5jaWEgbWVkaWEgcGFyZWFkYSBwb3Igc2VtaWxsYSBjb24gSUM5NSB0OyBib290c3RyYXAgcG9yIGdydXBvIGRlIHZhbGlkYWNpw7NuICgyMDAwIHJlbXVlc3RyZW9zKSBkZWwgRGljZSBtZWRpbyBlbiBsYXMgNSBzZW1pbGxhczsgVE9TVCBjb24gwrHOtC4iLA0KICAgICJjb250cmFzdGVfc2VjdW5kYXJpbyI6ICJHYW5hZG9yYSBmcmVudGUgYSBsYXMgNSBjb3JyaWRhcyBvcmlnaW5hbGVzIGRlbCBiZW5jaG1hcmsgKHJlc3VsdGFkb3MvdmFsaWRhY2lvbl9wb3JfaW1hZ2VuX2UyLyBvIGF1ZGl0b3LDrWEpLCBzb2xvIGRlc2NyaXB0aXZvOiBsYSByZWNldGEgcHVlZGUgZGlmZXJpciBzaSBlbCBhanVzdGUgcHJldmlvIGNhbWJpYSBsYSB0YXNhLiIsDQogICAgImxlY3R1cmEiOiAiU2kgbGEgaGVyZWRhZGEgY2FlIGVuIGVsIGNvbmp1bnRvIGVxdWl2YWxlbnRlIGFsIGdhbmFkb3IsIGxhIGhlcmVuY2lhIHF1ZWRhIGp1c3RpZmljYWRhIHBvciBsYSBiw7pzcXVlZGEuIFNpIGVsIGdhbmFkb3IgbGEgc3VwZXJhIHBvciBtw6FzIGRlIM60LCBzZSBkZWNsYXJhIHF1ZSBlbCBiZW5jaG1hcmsgc3ViZXN0aW3DsyBlc2EgZmFtaWxpYSB5IHNlIHJlcG9ydGEgbGEgZGlmZXJlbmNpYSwgc2luIGNhbWJpYXIgbGFzIGNvbmNsdXNpb25lcyBkZWwgdGVzdCB5YSBjZXJyYWRvLiINCiAgfSwNCiAgImFuYWxpc2lzX2VzcGFjaW8iOiB7DQogICAgImNvbnRyb2xfZ3VpYSI6ICIxNC42IiwNCiAgICAic3VwZXJmaWNpZSI6ICJNYXBhIGRlIGNhbG9yIGRlbCBEaWNlIGRlIGxhIGZhc2UgMSBzb2JyZSAodzEsIHcyKSBwb3IgdzMsIHVubyBwb3IgcGF0csOzbiBkZSBkcm9wb3V0IChmaWd1cmFzIGVuIGxvY2FsKSIsDQogICAgIm1vZGVsb19saW5lYWwiOiAieSA9IGIwICsgc3VtX2kgc3VtX3cgYmV0YV9pdyAxW3dfaSA9IHddICsgc3VtX2kgZ2FtbWFfaSBiX2kgKyBzdW1fe2k8an0gbGFtYmRhX2lqIGJfaSBiX2ogKyBlLCBwb3IgZmFtaWxpYSwgY29uIElDOTUgeSBwcm9wb3JjacOzbiBkZSB2YXJpYW56YSAoc3VtYSBkZSBjdWFkcmFkb3MgdGlwbyBJSSkiLA0KICAgICJwcnVlYmFzX2ZhY3RvcmVzIjogIkYgcGFyY2lhbCBwb3IgZmFjdG9yIChhbmNob3MgZGUgY2FkYSBjYXBhLCBkcm9wb3V0IGRlIGNhZGEgcG9zaWNpw7NuLCBpbnRlcmFjY2lvbmVzIGRlIGRyb3BvdXQpIGNvbiBIb2xtIHBvciBlbCBuw7ptZXJvIGRlIGZhY3RvcmVzOyB0ZW5kZW5jaWEgc29icmUgbG9nMiB3IHBvciBjYXBhIiwNCiAgICAiZHJvcG91dCI6ICJXaWxjb3hvbiBwYXJlYWRvIGVudHJlICgwLDAsMCkgeSBsYSBtZWRpYSBkZSBsb3MgNyBwYXRyb25lcyBjb24gZHJvcG91dCwgY29uIGxhcyAxMjUgY29tYmluYWNpb25lcyBkZSBhbmNob3MgY29tbyBibG9xdWVzOyB0YW1hw7FvIGRlIGVmZWN0byByID0gWi9zcXJ0KG4pIiwNCiAgICAicGFyZXRvIjogIkRpY2UgZGUgdmFsaWRhY2nDs24gZnJlbnRlIGEgcGFyw6FtZXRyb3MgeSBmcmVudGUgYSBzZWd1bmRvczsgYXJxdWl0ZWN0dXJhcyBubyBkb21pbmFkYXMiLA0KICAgICJzZW5zaWJpbGlkYWQiOiAiRGlzcGVyc2nDs24gZW50cmUgYXJxdWl0ZWN0dXJhcyAocmFuZ28sIERFKSBmcmVudGUgYSBsYSBERSBwb3Igc2VtaWxsYSAoZmFzZSAyIHkgcmVwZXRpY2nDs24pLiBTaSBzb24gcGFyZWNpZGFzLCBsYSBhcnF1aXRlY3R1cmEgZXMgaXJyZWxldmFudGUgcGFyYSBlc3RlIHByb2JsZW1hIHkgc2UgY29uY2x1eWUgYXPDrS4iLA0KICAgICJzb2JyZWFqdXN0ZSI6ICJCcmVjaGEgdHJhaW4tdmFsaWRhY2nDs24gZGUgbGEgZmFzZSAyIHBvciBmaW5hbGlzdGE7IGVuIGxhIGZhc2UgMSwgcMOpcmRpZGEgZGUgdmFsaWRhY2nDs24gbWVub3MgcMOpcmRpZGEgZGUgZW50cmVuYW1pZW50byBlbiBsYSDDumx0aW1hIMOpcG9jYSwgcG9yIGFuY2hvIHRvdGFsIHkgcGF0csOzbiBkZSBkcm9wb3V0IiwNCiAgICAiYm9yZGUiOiAiU2UgaW5mb3JtYSBzaSBlbCBnYW5hZG9yIGNhZSBlbiBlbCBib3JkZSBkZWwgZXNwYWNpbyAoYWxnw7puIHdfaSA9IDI1NiBvID0gMTYpIiwNCiAgICAiY29ycmVsYWNpb24iOiAiU3BlYXJtYW4gZW50cmUgZWwgRGljZSBkZSBsYSBmYXNlIDEgeSBsYSBtZWRpYSBkZSBsYSBmYXNlIDIgZW4gbG9zIGZpbmFsaXN0YXMiDQogIH0sDQogICJjb21wdXRvIjogew0KICAgICJmb3JtdWxhIjogIkNfdG90YWwgPSBtICh8QXwgdDEgKyAoayArIDIpIFIgdDIpICsgYWp1c3RlIHByZXZpbyArIHJlcGV0aWNpw7NuOyBtID0gMywgfEF8ID0gMTAwMCwgUiA9IDUiLA0KICAgICJlc3RpbWFjaW9uIjogInJlc3VsdGFkb3MvYnVzcXVlZGFfYXJxdWl0ZWN0dXJhX2UyL2VzdGltYWNpb25fY29zdG9fbG9jYWwuanNvbiAobG9jYWwpIHkgZXN0aW1hY2lvbl9jb21wdXRvLmpzb24gZGVsIHBpbG90byBlbiBMNCINCiAgfQ0KfQ0K"
},
"config/arquitecturas_compactas_e2.csv": {
"sha256": "3e510bc2ce4212d6077c21c64365ffd86fad6729e8c4453a79f2196b5bdb0eda",
"b64": "..."
}
}
for rel, item in INCRUSTADOS.items():
    data = base64.b64decode(item['b64'])
    assert hashlib.sha256(data).hexdigest() == item['sha256'], rel
    (PROJECT/rel).parent.mkdir(parents=True, exist_ok=True)
    (PROJECT/rel).write_bytes(data)
    print('escrito', rel, item['sha256'][:12])

## Entorno aislado, el mismo torch del benchmark
Python 3.11 fuera del kernel, torch 2.5.1+cu121 y `requirements-e2.txt`. Si `/content` se borró
(sesión nueva), esta celda lo reconstruye. Al final guarda el `pip freeze` en Drive.

In [ ]:
subprocess.run([sys.executable,'-m','pip','install','uv==0.6.9'],check=True)
ENV=Path('/content/imo-e2-env')
if not (ENV/'bin/python').exists():
    subprocess.run([sys.executable,'-m','uv','venv','--python','3.11',str(ENV)],check=True)
PY=str(ENV/'bin/python')
def pip_install(*items):
    subprocess.run([sys.executable,'-m','uv','pip','install','--python',PY,*items],check=True)
pip_install('setuptools==75.8.0')
pip_install('torch==2.5.1','torchvision==0.20.1','--index-url','https://download.pytorch.org/whl/cu121')
pip_install('-r',str(PROJECT/'requirements-e2.txt'))
subprocess.run([PY,'-c','import torch; assert torch.cuda.is_available(); print(torch.__version__,torch.cuda.get_device_name(0))'],check=True)
freeze=subprocess.check_output([sys.executable,'-m','uv','pip','freeze','--python',PY],text=True)
(OUTPUT/'environment.freeze.txt').write_text(freeze)

## Lanzador
`correr` ejecuta un paso en un proceso. `correr_paralelo` lanza `n` procesos a la vez en la misma GPU,
cada uno con un fragmento disjunto de las tareas (`--fragmento i/n`); las redes compactas no llenan una
L4, así que varios procesos aprovechan el tiempo muerto sin cambiar nada del protocolo (cada corrida es
la misma que en serie). Cada proceso deja su log en `logs/` dentro de Drive. `progreso()` cuenta las
corridas terminadas.

In [ ]:
RUNNER = str(PROJECT/'scripts/busqueda_arquitectura_e2.py')
def _comando(bloque, *argumentos):
    return [PY, '-u', RUNNER, bloque, *map(str, argumentos), '--output', str(OUTPUT)]
def correr(bloque, *argumentos):
    sello = datetime.datetime.now().strftime('%Y%m%dT%H%M%S')
    log = OUTPUT/'logs'/f'{bloque}_{sello}.log'
    comando = ' '.join(_comando(bloque, *argumentos))
    print('$', comando, '\n  log:', log)
    subprocess.run(['bash','-o','pipefail','-c',f'cd {PROJECT}/scripts && {comando} 2>&1 | tee -a {log}'],check=True)
def correr_paralelo(bloque, n, *argumentos):
    sello = datetime.datetime.now().strftime('%Y%m%dT%H%M%S'); procesos = []
    for i in range(n):
        log = OUTPUT/'logs'/f'{bloque}_{sello}_p{i}de{n}.log'
        f = open(log, 'a')
        procesos.append((subprocess.Popen(_comando(bloque, *argumentos, '--fragmento', f'{i}/{n}'),
                                          cwd=str(PROJECT/'scripts'), stdout=f, stderr=subprocess.STDOUT), f, log))
    print(f'{n} procesos de {bloque}; logs en', OUTPUT/'logs')
    try:
        while any(p.poll() is None for p, _, _ in procesos):
            time.sleep(120); progreso()
    finally:
        for p, f, _ in procesos:
            if p.poll() is None: p.terminate()
            f.close()
    fallidos = [str(log) for p, _, log in procesos if p.returncode != 0]
    if fallidos: raise RuntimeError(f'Procesos con error (ver logs): {fallidos}')
def progreso():
    t = {}
    for b in ('piloto','ajuste','fase1','fase2'):
        for r in (OUTPUT/b).glob('*/result.json'):
            fam = r.parent.name.split('_')[0]; t[(b, fam)] = t.get((b, fam), 0) + 1
    print(datetime.datetime.now().strftime('%H:%M'), {f'{b}:{f}': n for (b, f), n in sorted(t.items())}, flush=True)

## 1. Presupuesto, piloto y fidelidad de la fase 1
**Antes de correr el piloto**, escribe en `HORAS_FASE1` las horas de L4 que se aceptan para ajuste previo +
fase 1 + repetición (las acordadas al aprobar la búsqueda). El piloto corre una época de 5 anchos × 3
familias a 256 y a 128 px con la caché llena (unos 15-30 minutos estimados) y `estimar` deja
`estimacion_computo.json`. `fidelidad` aplica la regla fijada en el protocolo: la primera opción
(A: 256 px; B: 128 px; C: 128 px y mitad de train) cuya estimación cabe en `HORAS_FASE1`. Queda en
`fidelidad.json` y no se puede cambiar después de empezar el ajuste.

In [ ]:
HORAS_FASE1 = None   # p. ej. 40; se fija ANTES de ver el piloto
assert HORAS_FASE1 is not None, 'Fija el presupuesto de horas antes del piloto'
correr('piloto')
correr('estimar')
correr('fidelidad', '--horas-fase1', HORAS_FASE1)
print(json.dumps(json.loads((OUTPUT/'fidelidad.json').read_text())['elegida'], indent=1, ensure_ascii=False))

## 2. Ajuste previo de la tasa (9 corridas cortas)
Referencia (64,64,64)/(0,1,0), tasas 0,0003, 0,001 y 0,003, semilla 97, 6 épocas, a la fidelidad de la fase 1. Regla fijada antes:
se queda la de mayor Dice de validación salvo que gane a 0,001 por menos de 0,005.

In [ ]:
correr_paralelo('ajuste', 3)
correr('cerrar_ajuste')

## 3. Fase 1: las 1000 arquitecturas de cada familia
Semilla 97, 6 épocas, misma receta para todas. Las 40 primeras del orden se repiten con la semilla 98 para
estimar el ruido. El orden es una permutación fija: si hay que parar, lo hecho es una muestra aleatoria
del espacio, igual para las tres familias. `PROCESOS` es cuántos procesos comparten la GPU; mira
`nvidia-smi` en el primer progreso y súbelo si la utilización está baja y sobra memoria.

In [ ]:
EJECUTAR_FASE1 = False
PROCESOS = 4
if EJECUTAR_FASE1:
    correr_paralelo('fase1', PROCESOS)
progreso()

## 4. Finalistas
Calcula σ por semilla (repetición) y σ entre arquitecturas; k = 10 si σ_semilla/σ_arquitectura ≤ 1 en las
tres familias, k = 20 si no. Añade la referencia y la heredada. Solo se corre con la fase 1 completa;
con `PERMITIR_INCOMPLETA=True` se elige del prefijo hecho y queda registrado como desviación.

In [ ]:
PERMITIR_INCOMPLETA = False
correr('finalistas', *(['--permitir-incompleta'] if PERMITIR_INCOMPLETA else []))
print(json.dumps({f: v['fase2'] for f, v in json.loads((OUTPUT/'finalistas.json').read_text())['familias'].items()}, indent=1))

## 5. Fase 2: finalistas con 5 semillas y presupuesto completo
Semillas 11, 23, 37, 53, 71, hasta 30 épocas con paciencia 5; guarda el mejor checkpoint y mide la brecha
train-validación. Las tareas van por semilla: si se corta, todos los finalistas tienen las mismas semillas.

In [ ]:
EJECUTAR_FASE2 = False
PROCESOS = 4
if EJECUTAR_FASE2:
    correr_paralelo('fase2', PROCESOS)
progreso()

## 6. Resumen
Escribe `RESUMEN.json`, `RESUMEN.md` y `fase1.csv` en la carpeta de Drive. Descarga la carpeta completa
**sin** los `best.pt` (se quedan en Drive) para el análisis local en `resultados/busqueda_arquitectura_e2/`.

In [ ]:
subprocess.run([PY, str(PROJECT/'scripts/resumir_busqueda_arquitectura_e2.py'), '--output', str(OUTPUT)], check=True)
abiertas = sorted(p.parent.name for p in OUTPUT.glob('fase2/*/last.pt'))
print('Corridas de la fase 2 empezadas sin cerrar:', abiertas or 'ninguna')
# Paquete liviano para descargar (sin checkpoints):
import shutil
liviano = Path('/content/busqueda_arquitectura_e2_resultados.zip')
with zipfile.ZipFile(liviano, 'w', zipfile.ZIP_DEFLATED) as z:
    for p in OUTPUT.rglob('*'):
        if p.is_file() and p.suffix != '.pt':
            z.write(p, p.relative_to(OUTPUT))
shutil.copy(liviano, OUTPUT/liviano.name)
print('Descarga', OUTPUT/liviano.name, round(liviano.stat().st_size/1e6, 1), 'MB')